# 1 · A sales-order graph in LanceDB

This is the first of four notebooks. In this one we build a property graph of sales orders,
store it in LanceDB, and query it with Cypher through lance-csr. There's no hyperdimensional
computing (HDC) yet. We start with the graph because it's the **explicit** way to model
connected data, and it will be our source of truth in every notebook that follows.

The idea the four notebooks build towards:

> **A graph models connections explicitly. Hypervectors model them implicitly. Both answer
> questions about connected data, and they're stronger together.**

## The scenario

Imagine that a London homeware retailer ships products from three depots: **East London** (Barking), **West London**
(Park Royal) and the **Midlands**. It sells across the UK, and to customers in Ireland and
France, which since Brexit means customs paperwork on every EU order.

Next, a sales operations colleague gets a call from a customer in Ireland complaining that order **SO-1042** hasn't
arrived. The sales person is working with **partial information**. Here's what they know, and what they don't.

| Known | Unknown |
|---|---|
| it's a **lamp** (a cordless table lamp) | the destination **region** (that's in a different system) |
| it shipped from the **East London** depot | what went wrong |
| it's going to **Ireland** | |

They also remember handling a similar problem a few weeks ago, but not which order it was.
Finding that case would tell them what to check first.

## Why a graph?

The facts about a sales order live in different departmental views: the order system knows the
items, fulfilment knows the depot and the destination, and customer records know who placed
it. A graph joins these views with **explicit edges**, so you can start at the order and
follow the links.

Here's the shape of our graph: 8 node labels and 7 relationship types. Every past order that
went wrong also has a `HAD_ISSUE` edge, which carries a `note` property saying what happened.

![Sales-order graph schema](../img/sales-order-schema.png)

*The sales-order schema from slide 12 of the workshop presentation.*

And here are two orders from it, side by side: the open order SO-1042, and SO-0917, a past
order. They're different orders, for different customers in different Irish regions, but they
share four nodes: `cordless-table-lamp`, `lamps`, `east-london` and `ireland`.

![Shared graph pattern for SO-1042 and SO-0917](../img/shared-order-pattern.png)

That shared shape is what our colleague half-remembers. By the end of this notebook, we'll
find SO-0917 with a Cypher query.

## Setup

In [1]:
import lancedb
import polars as pl
from lance_csr import CypherEngine, GraphConfig

pl.Config.set_fmt_str_lengths(200)  # print the HAD_ISSUE notes in full

polars.config.Config

## The data

A labeled property graph (LPG) has two parts: **nodes**, grouped by node label, and **edges**,
grouped by relationship type. The full data is about 100 lines of Python literals, so it lives
in `sales_orders.py`, next to this notebook. Each node is a dict with an integer `id` and a
readable `name` (orders also have a `status`), and each edge is a `(src, dst)` pair of node
ids. The ids are numbered by label, so they're easy to decode: products are 2xx, depots 4xx,
countries 6xx, and an order's id is its order number. We'll see why they're integers when we
build the graph engine.

In [2]:
from sales_orders import NODES, EDGES

{label: len(rows) for label, rows in NODES.items()}

{'Customer': 6,
 'Order': 13,
 'Product': 8,
 'Category': 4,
 'Depot': 3,
 'Region': 11,
 'Country': 3,
 'Issue': 3}

In [3]:
{rel: len(pairs) for rel, pairs in EDGES.items()}

{'PLACED': 13,
 'CONTAINS': 13,
 'IN_CATEGORY': 8,
 'SHIPS_FROM': 13,
 'SHIPS_TO': 13,
 'IN_COUNTRY': 11,
 'HAD_ISSUE': 12}

Here's every edge that touches SO-1042, with each id swapped for its node's name. Note that there's no `HAD_ISSUE` edge yet, because
nobody knows what went wrong.

In [4]:
name = {n["id"]: n["name"] for rows in NODES.values() for n in rows}

for rel, pairs in EDGES.items():
    for edge in pairs:
        if 1042 in edge[:2]:
            print(f"{name[edge[0]]:18} -[{rel}]-> {name[edge[1]]}")

Liffey Interiors   -[PLACED]-> SO-1042
SO-1042            -[CONTAINS]-> cordless-table-lamp
SO-1042            -[SHIPS_FROM]-> east-london
SO-1042            -[SHIPS_TO]-> dublin


`HAD_ISSUE` edges have a third field, the `note`, which becomes an **edge property**.

In [5]:
EDGES["HAD_ISSUE"][:3]

[(901,
  701,
  'Commercial invoice missing; Irish customs held the parcel until the broker resent it.'),
 (904,
  701,
  'Wrong commodity code on the customs declaration; corrected and re-filed.'),
 (908, 701, 'Import VAT unpaid; released after the customer paid it.')]

## Store it in LanceDB

[LanceDB](https://docs.lancedb.com/) is an embedded vector retrieval library: in open source,
it's a directory on disk, with no server to run. We store the graph the way a relational
engine would, as **tables**.

Each node label gets its own table, and each relationship type gets its own table with `src`
and `dst` columns, plus a `note` column for `HAD_ISSUE`. `mode="overwrite"` means you can
re-run this cell as often as you like.

In [6]:
db = lancedb.connect("sales.lancedb")

for label, rows in NODES.items():
    db.create_table(label, rows, mode="overwrite")

for rel, edges in EDGES.items():
    if rel == "HAD_ISSUE":
        rows = [{"src": s, "dst": d, "note": note} for s, d, note in edges]
    else:
        rows = [{"src": s, "dst": d} for s, d in edges]
    db.create_table(rel, rows, mode="overwrite")

db.list_tables().tables

['CONTAINS',
 'Category',
 'Country',
 'Customer',
 'Depot',
 'HAD_ISSUE',
 'IN_CATEGORY',
 'IN_COUNTRY',
 'Issue',
 'Order',
 'PLACED',
 'Product',
 'Region',
 'SHIPS_FROM',
 'SHIPS_TO']

## Peek at an edge table

LanceDB tables use Apache Arrow's type system, so any table reads straight into a PyArrow
table. A relationship table is just an **edge list**: `src` and `dst` hold node ids, and any
extra columns are the edge's properties.

In [7]:
db.open_table("HAD_ISSUE").to_arrow().slice(0, 3)

pyarrow.Table
src: int64
dst: int64
note: string
----
src: [[901,904,908]]
dst: [[701,701,701]]
note: [["Commercial invoice missing; Irish customs held the parcel until the broker resent it.","Wrong commodity code on the customs declaration; corrected and re-filed.","Import VAT unpaid; released after the customer paid it."]]

Each LanceDB table is a **Lance dataset** on disk, and `to_lance()` hands that dataset to
lance-csr. lance-csr reads it directly, pushing filters into the scan, so there's no export or
copy between storage and the graph engine.

In [8]:
datasets = {name: db.open_table(name).to_lance() for name in (*NODES, *EDGES)}
{name: ds.count_rows() for name, ds in datasets.items()}

{'Customer': 6,
 'Order': 13,
 'Product': 8,
 'Category': 4,
 'Depot': 3,
 'Region': 11,
 'Country': 3,
 'Issue': 3,
 'PLACED': 13,
 'CONTAINS': 13,
 'IN_CATEGORY': 8,
 'SHIPS_FROM': 13,
 'SHIPS_TO': 13,
 'IN_COUNTRY': 11,
 'HAD_ISSUE': 12}

## Describe the graph's shape

The tables alone don't say which ones are nodes and which are edges. A `GraphConfig` tells
lance-csr: for each node label, the column that holds the node ID; for each relationship
type, its source and target node labels and the columns that hold their IDs. It plays the same
role as a schema in a graph database.

In [9]:
from sales_orders import REL_ENDPOINTS

REL_ENDPOINTS

{'PLACED': ('Customer', 'Order'),
 'CONTAINS': ('Order', 'Product'),
 'IN_CATEGORY': ('Product', 'Category'),
 'SHIPS_FROM': ('Order', 'Depot'),
 'SHIPS_TO': ('Order', 'Region'),
 'IN_COUNTRY': ('Region', 'Country'),
 'HAD_ISSUE': ('Order', 'Issue')}

In [10]:
builder = GraphConfig.builder()
for label in NODES:
    builder = builder.node_table(label, "id")
for rel, (src_label, dst_label) in REL_ENDPOINTS.items():
    builder = builder.rel_table(rel, src_label, dst_label, "src", "dst")

cfg = builder.build()
cfg

GraphConfig(nodes=["order", "depot", "issue", "product", "country", "customer", "category", "region"], relationships=["had_issue", "ships_from", "contains", "in_category", "ships_to", "placed", "in_country"])

## Build the graph engine

A `CypherEngine` holds the config and the datasets, and runs Cypher queries. With
`use_csr=True`, it also builds a **CSR** (compressed sparse row) adjacency for each
relationship type. The edges are sorted by source node, so each node's neighbours sit next to
each other in one array, and a second array of offsets says where each node's run starts.
Following an edge becomes two array lookups instead of a join, which is what makes multi-hop
patterns fast on large graphs.

lance-csr builds a CSR only over **integer** node ids. That's why every node has an integer
`id`, with its readable `name` as a separate property. The stats show one CSR per relationship
type, with its edge count.

In [11]:
engine = CypherEngine(cfg, datasets, use_csr=True)
{rel: stats["edges"] for rel, stats in sorted(engine.csr_stats().items())}

{'CONTAINS': 13,
 'HAD_ISSUE': 12,
 'IN_CATEGORY': 8,
 'IN_COUNTRY': 11,
 'PLACED': 13,
 'SHIPS_FROM': 13,
 'SHIPS_TO': 13}

## A first Cypher query: one hop

"What's in SO-1042?" The pattern `(o:Order {name: 'SO-1042'})-[:CONTAINS]->(p:Product)` follows
one `CONTAINS` edge out of the order. The result is an Arrow table, which we turn into a
polars DataFrame to show it.

In [12]:
q = "MATCH (o:Order {name: 'SO-1042'})-[:CONTAINS]->(p:Product) RETURN p.name"
pl.from_arrow(engine.execute(q))

p.name
str
"""cordless-table-lamp"""


## Traversing the departmental paths

Now the whole picture of SO-1042 in one query: who placed it, what's in it, where it ships
from and where it's going. A `MATCH` can hold several patterns separated by commas. They all
share the variable `o`, so they all describe the same order. This is the graph version of a
five-hop path traversal across three departments' tables.

In [13]:
q = '''
MATCH (c:Customer)-[:PLACED]->(o:Order {name: 'SO-1042'}),
      (o)-[:CONTAINS]->(p:Product)-[:IN_CATEGORY]->(cat:Category),
      (o)-[:SHIPS_FROM]->(d:Depot),
      (o)-[:SHIPS_TO]->(r:Region)-[:IN_COUNTRY]->(k:Country)
RETURN c.name AS customer, p.name AS product, cat.name AS category, d.name AS depot, r.name AS region, k.name AS country
'''
pl.from_arrow(engine.execute(q))

customer,product,category,depot,region,country
str,str,str,str,str,str
"""Liffey Interiors""","""cordless-table-lamp""","""lamps""","""east-london""","""dublin""","""ireland"""


The graph knows the region is `dublin`. Our colleague doesn't, because they're looking at a
different screen. We'll keep that gap in the story: below, the region is left out of the
question.

## What lance-csr does under the hood

There's no separate graph store here. lance-csr compiles a Cypher pattern into a query plan
that Apache DataFusion runs over the Lance datasets. Here's the logical plan for the one-hop
query from before. Read it from the bottom up: scan the `Order` table, filter to SO-1042, then
a `CsrExpand` follows its `CONTAINS` edges through the CSR and fetches the products by row
position. There's no join in the plan.

In [14]:
q = "MATCH (o:Order {name: 'SO-1042'})-[:CONTAINS]->(p:Product) RETURN p.name"

section = None
for line in engine.explain(q).splitlines():  # explain() returns a text table; print its logical_plan rows
    if line.startswith("|"):
        _, kind, text, _ = line.split("|")
        section = kind.strip() or section
        if section == "logical_plan":
            print(text.rstrip())

 Projection: p__name AS p.name
   CsrExpand: key=o__id -> [contains_1__src, contains_1__dst], edges=13, take=p[p__id*, p__name]
     Projection: order.id AS o__id, order.name AS o__name, order.status AS o__status
       Filter: order.name = Utf8("SO-1042")
         TableScan: order


## Searching for a motif

A **motif** is a small, recurring pattern of nodes and edges: a shape you want to find
wherever it occurs in a graph. Our colleague's question, "have we seen an order like this
before?", is a motif search. The motif is an order that contains a lamp, ships from East
London, and goes to a region in Ireland.

We add one more pattern, `(o)-[h:HAD_ISSUE]->(i:Issue)`. It keeps only past orders that went
wrong (open orders have no `HAD_ISSUE` edge yet), and it returns the edge's `note`. This is the
query we'll come back to in notebook 3.

In [15]:
motif_query = '''
MATCH (o:Order)-[:CONTAINS]->(:Product)-[:IN_CATEGORY]->(:Category {name: 'lamps'}),
      (o)-[:SHIPS_FROM]->(:Depot {name: 'east-london'}),
      (o)-[:SHIPS_TO]->(:Region)-[:IN_COUNTRY]->(:Country {name: 'ireland'}),
      (o)-[h:HAD_ISSUE]->(i:Issue)
RETURN o.name, i.name, h.note
'''
result = engine.execute(motif_query)
pl.from_arrow(result)

o.name,i.name,h.note
str,str,str
"""SO-0917""","""dispatch_hold""","""Cordless lamps (lithium batteries) held at East London depot for dangerous-goods paperwork; released after calling the depot."""


In [16]:
rows = [(r["o.name"], r["i.name"]) for r in result.to_pylist()]
assert rows == [("SO-0917", "dispatch_hold")], rows

There's the remembered case. SO-0917 was a **dispatch hold**: cordless lamps contain lithium
batteries, and the East London depot held them for dangerous-goods paperwork. Our colleague
now has a specific question for the depot.

## Exact answers to exact questions

A graph query is precise. It returns exactly the orders that fit the pattern, and nothing
else. That's what you want for an audit, but it has a cost when you only *roughly* know what
you're looking for.

Suppose our colleague had looked up the region and added it to the motif. SO-0917 went to
Cork, not Dublin, so the precedent disappears.

In [17]:
stricter = motif_query.replace("(o)-[:SHIPS_TO]->(:Region)", "(o)-[:SHIPS_TO]->(:Region {name: 'dublin'})")
engine.execute(stricter).num_rows

0

The opposite problem: orders that match *some* of the motif are invisible to the motif query.
Below, we drop the lamps condition. SO-0917 comes back, and so do two orders that also shipped
from East London to Ireland but weren't lamps. To see near matches like these, you'd have to
write a query for every combination of conditions you're willing to drop. With three
conditions, that's already seven queries.

In [18]:
q = '''
MATCH (o:Order)-[:SHIPS_FROM]->(:Depot {name: 'east-london'}),
      (o)-[:SHIPS_TO]->(:Region)-[:IN_COUNTRY]->(:Country {name: 'ireland'}),
      (o)-[:CONTAINS]->(:Product)-[:IN_CATEGORY]->(c:Category),
      (o)-[:HAD_ISSUE]->(i:Issue)
RETURN o.name, c.name, i.name
'''
pl.from_arrow(engine.execute(q))

o.name,c.name,i.name
str,str,str
"""SO-0901""","""sofas""","""customs_delay"""
"""SO-0920""","""desks""","""customs_delay"""
"""SO-0917""","""lamps""","""dispatch_hold"""


## Conclusions

In this notebook, we stored a sales-order graph in LanceDB and queried it with Cypher. The
graph models connections **explicitly**: every edge is a row, and every answer can be traced
back to the rows that produced it. The motif query found the precedent SO-0917 exactly.

It also showed the limits of exact matching. Add a detail and the precedent vanishes; drop
one and you need a new query for each combination. In the next notebook we meet a different
representation, **hypervectors**, where a whole pattern becomes a single list of numbers.
Similar patterns get similar hypervectors, so "find orders like this one" becomes a vector
search.